In [1]:
# --- Cell 1: Ingest Data and Setup Test Environment ---
import sys
import os
import pandas as pd

sys.path.append(os.path.abspath(os.path.join('..')))
from src.data_loader import InsuranceDataLoader
from src.hypothesis_tests import InsuranceHypothesisTester

# Load text data matrix
loader = InsuranceDataLoader(file_path="../data/MachineLearningRating_v3.txt")
df = loader.load_and_initialize()

# Setup explicit binary claim frequency flag
df['has_claim'] = df['TotalClaims'] > 0

tester = InsuranceHypothesisTester(df)
summary_metrics = []
print(f"Ecosystem running. Evaluating {len(df):,} insurance transaction rows...")

Ecosystem running. Evaluating 1,000,098 insurance transaction rows...


In [2]:
# --- Cell 2: Run Statistical Verification Pipelines ---
# 1. Risk vs Province
_, p_prov = tester.run_chi_square_test(feature='Province')
summary_metrics.append({"Hypothesis": "Risk is distributed equally across Provinces", "Test": "Chi-Square", "P-Value": p_prov})

# 2. Risk vs Postal Code
_, p_post = tester.run_chi_square_test(feature='PostalCode')
summary_metrics.append({"Hypothesis": "Risk is distributed equally across Zip Codes", "Test": "Chi-Square", "P-Value": p_post})

# 3. Margin Differences Between Top 2 Active Postal Codes
top_zips = df['PostalCode'].value_counts().index[:2].tolist()
_, p_margin = tester.run_two_sample_ttest(segment_col='PostalCode', group_a_val=top_zips[0], group_b_val=top_zips[1], target_kpi='Margin')
summary_metrics.append({"Hypothesis": f"Margin is equal between Zip {top_zips[0]} and {top_zips[1]}", "Test": "Two-Sample T-Test", "P-Value": p_margin})

# 4. Claim Severity Profiles Across Genders
_, p_gender = tester.run_two_sample_ttest(segment_col='Gender', group_a_val='Male', group_b_val='Female', target_kpi='TotalClaims')
summary_metrics.append({"Hypothesis": "Claim severity is equal between Men and Women", "Test": "Two-Sample T-Test", "P-Value": p_gender})

# Display clean executive table layout
print("\n📊 TASK 2 LAB EXPERIMENT RESULTS:")
pd.DataFrame(summary_metrics)


📊 TASK 2 LAB EXPERIMENT RESULTS:


,Hypothesis,Test,P-Value
0,Risk is distributed equally across Provinces,Chi-Square,5.925511e-19
1,Risk is distributed equally across Zip Codes,Chi-Square,3.152172e-30
2,Margin is equal between Zip 2000 and 122,Two-Sample T-Test,2.444624e-01
3,Claim severity is equal between Men and Women,Two-Sample T-Test,5.680287e-01
